# Contingency Analysis in Transmission Grids

Contingency analysis is a critical process in power system operations used to assess the impact of potential failures (e.g., line outages) on grid stability and reliability. It helps operators prepare for unexpected events by simulating scenarios such as N-1 or N-2 contingencies, where one or more components are removed from service. This analysis ensures that the grid can continue to operate within safe limits even under stressed conditions.

---

## Dataset Generation and Model Evaluation

The dataset used in this study originates from the Texas transmission grid, which includes approximately 2,000 nodes. Using the contingency mode of the `gridfm-datakit`, we simulated N-2 contingencies by removing up to two transmission lines at a time. For each scenario, we first solved the optimal power flow (OPF) problem to determine the generation dispatch. Then, we applied the contingency by removing lines and re-solved the power flow to observe the resulting grid state.

This process generated around 100,000 unique scenarios. Our model, **GENCO**, was trained on this dataset to predict power flow outcomes. For demonstration purposes, we selected a subsample of 181 scenarios. The `gridfm-datakit` also computed DC power flow results, enabling a comparison between GENCO predictions and traditional DC power flow estimates, specifically in terms of line loading accuracy.

All predictions are benchmarked against the ground truth obtained from AC power flow simulations. Additionally, we analyze bus voltage violations, which GridFM can predict but are not captured by the DC solver, highlighting GENCO’s enhanced capabilities in modeling grid behavior.


In [ ]:
# Only run this cell if running on Colab.
import sys

if "google.colab" in sys.modules:
    try:
        #!git clone https://github.com/gridfm/gridfm-graphkit.git   CHANGE IF WE PUT THIS NOTEBOOK ON MAIN
        !git clone -b normalization_fixed_eval_predict https://github.com/gridfm/gridfm-graphkit.git
        !pip install ./gridfm-graphkit
    except Exception as e:
        print(f"Failed to start Google Collab setup, due to {e}")

# If running on Colab you will be asked to restart the session. This is normal and you should do so.


In [ ]:
%cd gridfm-graphkit/examples/notebooks/
%pwd

In [21]:
import os
import sys
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from contingency_utils import *
from pathlib import Path
sys.path.append("../")

if 'google.colab' in sys.modules:
  import gdown
  from google.colab import output
  output.enable_custom_widget_manager()

## Load Data

We load both the ground truth and predicted values of the power flow solution. The predictions are generated using the `gridfm-graphkit` CLI:

```bash
gridfm-graphkit predict ...
```

We then merge the datasets using `scenario` and `bus` as keys, allowing us to align the predicted and actual values for each grid state and bus.

In [22]:
root_pred_folder = "../data/contingency/"
prediction_label = "predictionsv1"
label_plot = "GENCO"
data_path = Path("../data/contingency/")
data_path.mkdir(parents=True, exist_ok=True)

Download the sample data provided for this tutorial

In [ ]:
if 'google.colab' in sys.modules:
  try:
    !gdown 19gOdmVHou4NTbIch7tjtjynQpQHg1sLG -O ../data/contingency/texas_nminus2_small.zip
  except:
    print(f"Download failed via g.down!")
    # full data path
    # https://drive.google.com/file/d/19gOdmVHou4NTbIch7tjtjynQpQHg1sLG/view?usp=sharing

!unzip ../data/contingency/texas_nminus2_small.zip -d ../data/contingency/

In [ ]:
preds = pd.read_parquet(os.path.join(root_pred_folder, "filtered_preds_light2.parquet"))
bus_data = pd.read_parquet(os.path.join(root_pred_folder, "filtered_bus_data_light2.parquet"))
branch_data = pd.read_parquet(os.path.join(root_pred_folder, "filtered_branch_data_light2.parquet"))

In [ ]:
# Create one df for GT and preds
pf_node = preds.merge(bus_data, on=["scenario", "bus"], how="left")
pf_node.head()

In [ ]:
# Build GENCO/DC state variants used to compute comparable branch flows
pf_node["Vm_pred_corrected"] = pf_node["Vm_pred"].astype(np.float64)
pf_node["Va_pred_corrected"] = pf_node["Va_pred"]
pf_node["Vm_dc_corrected"] = np.ones(len(pf_node), dtype=np.float64)
pf_node["Va_dc_corrected"] = pf_node["Va_dc"]
# Branch-power function expects degree inputs
pf_node['Va_pred_corrected'] = np.rad2deg(pf_node['Va_pred_corrected'])
pf_node

## Compute branch current and line loading

In [6]:
# Compute per-branch loading for AC ground truth, GENCO prediction, and DC baseline.
rated_branch_data = branch_data[branch_data["rate_a"] > 0].copy()
# Stress-test: reducing thermal limits increases normalized loading values.
rate_a = rated_branch_data["rate_a"].to_numpy(dtype=np.float64) / 1.25

loadings_map = {}
for flag in ("gt", "pred", "dc"):
    pf, qf, pt, qt = compute_branch_powers_vectorized(
        rated_branch_data,
        pf_node,
        sn_mva=100.0,
        flag=flag,
    )
    s_from = np.hypot(pf, qf)
    s_to = np.hypot(pt, qt)
    loadings_map[flag] = (np.maximum(s_from, s_to) / rate_a).flatten()

loadings = loadings_map["gt"]
loadings_genco = loadings_map["pred"]
loadings_dc = loadings_map["dc"]

In [7]:
loadings = loadings.flatten()
loadings_pred = loadings_genco.flatten()
loadings_dc = loadings_dc.flatten()

overloadings_mask = (loadings > 1.0)
overloadings_pred_mask = (loadings_pred > 0.988)
overloadings_dc_mask = (loadings_dc > 0.944)

## Compute metrics for overloading classification for GridFM and DC PF
- Below are the results of GENCO for overloading classification (HGNS version)
- Ground truth is AC PF.

In [ ]:
TP_genco, FP_genco, TN_genco, FN_genco = compute_cm_metrics(
    overloadings_mask, overloadings_pred_mask, prediction_label, label_plot
)

- Below are the results of DC PF for overloading classification
- Ground truth is AC PF.
- We use a threshold of 0.95 to make sure we identify all overloads

In [ ]:
TP_dc, FP_dc, TN_dc, FN_dc = compute_cm_metrics(
    overloadings_mask, overloadings_dc_mask, "DC", label_plot
)

## Histogram of true line loadings

In [ ]:
plt.hist(loadings, bins=100)
plt.xlabel("Line Loadings")
plt.ylabel("Frequency")
plt.title("Line loadings")
# log scale
plt.savefig(f"loadings_histogram_{prediction_label}.png")
plt.show()

## Predicted vs True line loading

In [ ]:
true_vals = loadings
gfm_vals = loadings_pred
dc_vals = loadings_dc

plot_mass_correlation_density(true_vals, gfm_vals, prediction_label, label_plot)

In [ ]:
plot_mass_correlation_density(true_vals, dc_vals, "DC", "DC Solver")

In [ ]:
plot_cm(TN_genco, FP_genco, FN_genco, TP_genco, prediction_label, label_plot)

In [ ]:
plot_cm(TN_dc, FP_dc, FN_dc, TP_dc, "DC", "DC Solver")

In [ ]:
# Histograms of loadings
plot_loading_predictions(
    loadings_pred,
    loadings_dc,
    loadings,
    prediction_label,
    label_plot,
)

## Voltage violations

In [ ]:
plot_mass_correlation_density_voltage(pf_node, prediction_label, label_plot, vm_nominal=1.0,
    vm_dev_threshold=0.075)

In [18]:
# Compute CM for Vm and Vm predicted
vm_dev_true = np.abs(pf_node["Vm"].to_numpy() - 1.0)
vm_dev_genco = np.abs(pf_node["Vm_pred_corrected"].to_numpy() - 1.0)
vm_violation_true = vm_dev_true > 0.075
vm_pred_genco = vm_dev_genco > 0.071

In [ ]:
TP_genco_vm, FP_genco_vm, TN_genco_vm, FN_genco_vm = compute_cm_metrics(
    vm_violation_true, vm_pred_genco, "Vm violations", label_plot
)

In [ ]:
plot_cm_vm(TN_genco_vm, FP_genco_vm, TP_genco_vm, TP_dc, "VM", "Vm Violation")
